# JalNetra — Data Exploration & Analysis

This notebook explores the generated training datasets for the three JalNetra ML models:
1. **Anomaly Detector** — Water quality contamination & sensor fault detection
2. **Depletion Predictor** — Groundwater level forecasting (30-day horizon)
3. **Irrigation Optimizer** — Crop-specific water scheduling

Data distributions are calibrated to match CPCB monitoring stations, India-WRIS groundwater records, and IMD meteorological data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data')
print('Available datasets:', [f.name for f in DATA_DIR.glob('*.parquet')])

## 1. Anomaly Detection Dataset

In [ ]:
anomaly_df = pd.read_parquet(DATA_DIR / 'anomaly_detection_train.parquet')
print(f'Shape: {anomaly_df.shape}')
print(f'Date range: {anomaly_df["timestamp"].min()} to {anomaly_df["timestamp"].max()}')
print(f'\nLabel distribution:')
print(anomaly_df['label'].value_counts().rename({0: 'Normal', 1: 'Contamination', 2: 'Sensor Fault'}))
anomaly_df.describe()

In [ ]:
# BIS IS 10500:2012 threshold visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

params = [
    ('tds', 'TDS (ppm)', [500, 2000], ['Acceptable', 'Critical']),
    ('ph', 'pH', [6.5, 8.5, 5.5, 9.5], ['Acceptable Low', 'Acceptable High', 'Critical Low', 'Critical High']),
    ('turbidity', 'Turbidity (NTU)', [1, 25], ['Acceptable', 'Critical']),
    ('do', 'Dissolved Oxygen (mg/L)', [6, 2], ['Acceptable', 'Critical']),
]

for ax, (col, title, thresholds, labels) in zip(axes.flat, params):
    colors = {0: '#2ecc71', 1: '#e74c3c', 2: '#f39c12'}
    for label_val, color in colors.items():
        subset = anomaly_df[anomaly_df['label'] == label_val][col]
        if len(subset) > 0:
            ax.hist(subset, bins=80, alpha=0.5, color=color, 
                    label=['Normal', 'Contamination', 'Sensor Fault'][label_val])
    for t in thresholds:
        ax.axvline(t, color='red', linestyle='--', alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()

plt.suptitle('Water Quality Parameter Distributions (BIS IS 10500:2012 Thresholds)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Groundwater Level Dataset

In [ ]:
gw_df = pd.read_parquet(DATA_DIR / 'groundwater_levels_train.parquet')
gw_df['date'] = pd.to_datetime(gw_df['date'])
print(f'Shape: {gw_df.shape}')
print(f'Wells: {gw_df["well_id"].nunique()}')
print(f'Date range: {gw_df["date"].min()} to {gw_df["date"].max()}')
gw_df.describe()

In [ ]:
# Groundwater level trends with monsoon recharge
fig, ax = plt.subplots(figsize=(16, 6))
for well_id in gw_df['well_id'].unique()[:5]:
    well = gw_df[gw_df['well_id'] == well_id]
    ax.plot(well['date'], well['water_level_m'], alpha=0.7, label=well_id)

ax.invert_yaxis()  # Depth increases downward
ax.set_ylabel('Water Level (meters below ground)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_title('Groundwater Level Trends — Depletion + Monsoon Recharge Cycles', fontsize=14, fontweight='bold')
ax.legend(title='Well ID')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

## 3. Irrigation Optimization Dataset

In [ ]:
irr_df = pd.read_parquet(DATA_DIR / 'irrigation_optimization_train.parquet')
print(f'Shape: {irr_df.shape}')
print(f'Crop types: {irr_df["crop_type"].unique()}')
irr_df.describe()

In [ ]:
# Water savings by crop type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

crop_eff = irr_df.groupby('crop_type')['efficiency_score'].mean().sort_values(ascending=True)
crop_eff.plot(kind='barh', ax=axes[0], color=sns.color_palette('viridis', len(crop_eff)))
axes[0].set_xlabel('Average Water Savings vs Flood Irrigation (%)')
axes[0].set_title('Water Efficiency by Crop Type', fontweight='bold')

axes[1].scatter(irr_df['soil_moisture_pct'], irr_df['irrigation_amount_liters'], 
                c=irr_df['temperature_c'], cmap='RdYlBu_r', alpha=0.1, s=2)
axes[1].set_xlabel('Soil Moisture (%)')
axes[1].set_ylabel('Recommended Irrigation (liters)')
axes[1].set_title('Irrigation vs Soil Moisture (colored by temperature)', fontweight='bold')
plt.colorbar(axes[1].collections[0], label='Temperature (°C)')

plt.tight_layout()
plt.show()

## 4. Feature Correlations

In [ ]:
# Anomaly detection feature correlations
numeric_cols = ['tds', 'ph', 'turbidity', 'do', 'flow_rate', 'water_level_m', 'tds_rate', 'ph_rate']
corr = anomaly_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Sensor Reading Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()